<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Apêndice D: Usando LLMs maiores

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- Os capítulos principais usam o modelo base Qwen3 0.6B porque ele é o menor modelo da
família Qwen3 e, portanto, o mais fácil de rodar em hardware de consumo
- No entanto, a mesma implementação de `Qwen3Model` do apêndice C também pode ser usada para carregar checkpoints densos maiores do Qwen3, com o mesmo código de modelo em PyTorch feito do zero

&nbsp;
## D.1 Configurações densas maiores do Qwen3

O repositório inclui dicionários de configuração para vários modelos densos maiores do Qwen3 (além do modelo 0.6B) em
`reasoning_from_scratch.appendix_c` ([reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)):

| Tamanho do modelo | Dicionário de configuração |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- Como mencionado na figura acima, estas são as variantes "densas" do Qwen3, que conseguem rodar em GPUs únicas
- Também existem variantes "esparsas" de Mixture-of-Experts do Qwen3, mas elas não são suportadas pelo código deste livro; no entanto, se você tem interesse em uma implementação do zero, pode encontrar uma aqui: https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- Todas elas usam o mesmo padrão geral de arquitetura do modelo 0.6B do apêndice C
- O que muda é o tamanho do embedding, o número de layers, o número de attention heads e
a dimensão oculta do feed-forward

- Como limite inferior aproximado, armazenar os pesos em bfloat16 exige cerca de 2 bytes por parâmetro
- Isso significa que só os pesos do checkpoint ficam na ordem de:

| Tamanho do modelo | Memória aproximada dos pesos em bfloat16 |
| --- | --- |
| 1.7B | cerca de 3,4 GB |
| 4B | cerca de 8 GB |
| 8B | cerca de 16 GB |
| 14B | cerca de 28 GB |
| 32B | cerca de 64 GB |


- Na prática, o uso real de memória em tempo de execução é maior, porque também precisamos de memória para
ativações, buffers temporários e, frequentemente, o KV cache

&nbsp;
## D.2 Visão geral do download de checkpoints maiores

- Diferente dos checkpoints 0.6B usados nos capítulos principais, os modelos Qwen3 oficiais maiores são
tipicamente distribuídos como arquivos `safetensors`, às vezes divididos em vários shards
- A função auxiliar `download_from_huggingface_from_snapshots`, usada para carregá-los, exige alguns pacotes adicionais:

```bash
!uv add huggingface_hub safetensors
```

ou

```bash
!pip install huggingface_hub safetensors
```

&nbsp;
## D.3 Carregando um modelo base maior

- Baixar os pesos:

In [2]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

Using Apple Silicon GPU (MPS)


/Users/sebastian/Developer/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 2616.79it/s]


- Inicializar o modelo:

In [3]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

Model uses weight tying.


Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

- Carregar o tokenizer:

In [4]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# Note that the original base tokenizer is called "tokenizer.json"
# We rename it to distinguish from the reasoning tokenizer (next section)
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- Usar o modelo:

In [5]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

 Large language models are artificial intelligence systems that use deep learning techniques to understand and generate human-like text. They are trained on vast amounts of data and can perform a wide range of natural language processing tasks, such as translation, summarization, and question answering.

&nbsp;
## D.4 Carregando uma variante de raciocínio maior

- A mesma ideia também funciona para modelos Qwen3 maiores no estilo de raciocínio
- A arquitetura para um dado tamanho de modelo permanece a mesma; só mudam o checkpoint e as configurações do tokenizer

Por exemplo, para carregar a variante de raciocínio 4B em vez da variante base 4B, faríamos o seguinte:

- trocar o ID do repositório de `Qwen/Qwen3-4B-Base` para `Qwen/Qwen3-4B`;
- copiar o arquivo `tokenizer.json` para `tokenizer-reasoning.json`;
- inicializar o tokenizer da seguinte forma:

```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- O restante do código de carregamento e uso do modelo permanece igual

&nbsp;
## D.5 Recomendações práticas

- Sem código nesta seção